# 1.1-GEMM
GEMM is short for General Matrix Multiplication.
GEMM是通用矩阵乘法的缩写。

$A\in R^{m\times k}, B\in R^{k\times n}, C = A \times B, C\in R^{m\times n}$

## Profiler

测试框架在仓库根目录 `test/` 包中（`config` / `gpu` / `ops` / `registry` / `verify` / `benchmark` / `plot` / `native`，见 `test/README.md`）。GEMM 通过本文件夹的 `op.py` 接入：注册实现、基线、原生 kernel 构建。notebook 只负责导入、调用与展示。

In [ ]:
# ═══════════════════════════════════════════════════════════
# GEMM Profiler — framework import & GPU selection
# ═══════════════════════════════════════════════════════════
# 测试框架在仓库根目录 test/ 包中（test/README.md）；GEMM 通过 op.py 接入。

import os
import sys
import pathlib

import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

# 把仓库根目录加入 sys.path（从 notebook 目录向上找包含 test/ 的目录）
_NB_FILE = globals().get("__vsc_ipynb_file__") or (globals().get("_dh") or [""])[0] or None
_d = pathlib.Path(_NB_FILE).parent if _NB_FILE else pathlib.Path(os.getcwd())
while not (_d / "test" / "ops.py").exists() and _d.parent != _d:
    _d = _d.parent
if str(_d) not in sys.path:
    sys.path.insert(0, str(_d))

from test import *
import test.config as config   # 可动态修改的配置

# 后端开关（默认在 test/config.py；也可在此动态覆盖）：
#   config.TEST_BACKENDS = "cpu"   # 仅 CPU
#   config.TEST_BACKENDS = "gpu"   # 仅 GPU

op = load_op("gemm")           # 加载 op.py：注册 Python 实现与基线
device = select_gpu()          # 自动选择空闲 GPU（CPU-only 模式下跳过）
if device is not None:
    from cupy.cuda import cublas
    print(f"cuBLAS version: {cublas.getVersion(device.cublas_handle)}")
print(f"CuPy: {cp.__version__}  |  NumPy: {np.__version__}")

## CPU Implementations

CPU 实现注册为 `backend="cpu"`：NumPy 数组，`perf_counter` + 自适应迭代计时。

NumPy BLAS 基线在 `op.py` 中注册；本 section 只放 notebook 里的教学实现。

### Python

In [ ]:
# ═══════════════════════════════════════════════════════════
# Python CPU Implementations
# ═══════════════════════════════════════════════════════════
# 实现签名统一为 fn(A, B) -> C（M/N/K 从 shape 推导）

if config.TEST_BACKENDS in ("all", "cpu"):

    @register_op(op.NAME, "NumPy einsum", backend="cpu", warmup=2)
    def numpy_einsum_gemm(A: np.ndarray, B: np.ndarray) -> np.ndarray:
        """np.einsum without the optimization path (pure einsum machinery)."""
        return np.einsum("ik,kj->ij", A, B, optimize=False)


    @register_op(op.NAME, "Python triple loop", backend="cpu",
                 max_size=128, warmup=0, iters=1)
    def python_naive_gemm(A: np.ndarray, B: np.ndarray) -> np.ndarray:
        """Pure-Python ijk loop — for education only, ~0.001 GFLOPS.

        Gated to sizes ≤ 128 (max_size) so it doesn't stall the sweep.
        """
        M, K = A.shape
        N = B.shape[1]
        C = np.zeros((M, N), dtype=DTYPE_NP)
        for i in range(M):
            for j in range(N):
                for k in range(K):
                    C[i,j] += A[i,k] * B[k, j]
        return C


    print(f"Registered: {list_implementations(op.NAME)}")
else:
    print("GPU-only mode: skipping Python CPU implementations.")

### C++

C++ 实现在 `gemm_cpu.cpp` 中；构建、加载与注册由 `op.py` 的 `prepare()` 自动完成（版本化产物 + mtime 缓存，清单在 op.py 的 `CPP_IMPLS`）。

Naive Implementations just use three loops.

TILING: Save how much?

suppose $A, B, C\in R^{N\times N}$, method ikj = $N^3+N^2$, tile stores the $B\times B$ block in the cache.

Then, tile will read $(N/B)^3\times 2B^2 = 2N^3 / B$

## GPU kernels

GPU 实现注册为 `backend="gpu"`（默认值）：数据为 CuPy 数组，计时用 CUDA Event。

Kernel 代码在 `gemm_gpu.cu` 中（nvcc 编译为版本化的 `gemm_gpu.<mtime>-<size>.so`）。`op.prepare()` 会检测源码是否比编译产物新，过期则自动重编译并重载——所以改完 `.cu` 只需直接重跑 Results cell。
`__global__` kernel 不能直接被 ctypes 调用，因此每个 kernel 配一个 `extern "C"` 的 host launcher，在调用方提供的 stream 上启动（保证 CUDA Event 计时正确）。

新 kernel 的接入方式：`.cu` 中写 kernel + launcher → 在 `op.py` 的 `CU_IMPLS` 清单中加一行 `("launcher 名", "显示名")`。

### Baselines

NumPy BLAS（CPU）与 cuBLAS / cuBLAS (cupy)（GPU）已由 `op.py` 在 `load_op` 时注册。
speedup 图的基线默认取 NumPy BLAS / cuBLAS；可覆盖（无需重启 kernel）：

```python
config.BASELINES = {"gemm": {"gpu": "cuBLAS (cupy)"}}
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# Native implementations — C++/CUDA build, load & register
# ═══════════════════════════════════════════════════════════
# 实现清单在 op.py（CPP_IMPLS / CU_IMPLS）。prepare() 按源码 mtime 检查过期
# 产物、自动重编译并重载、重新注册；Results cells 也会调用它——改完
# gemm_cpu.cpp / gemm_gpu.cu 后直接重跑 Results 即可。

op.prepare()

## Results

### Step 1: Verify Correctness

Run this cell to check the correctness of GEMM implementation. 
The baseline is NumPy FP32.

In [ ]:
# Verify all registered implementations
# 只验证某个后端：verify_all(op, backends="gpu")  # 或 "cpu"
# prepare() 会检查 .so 是否过期，过期则自动重编译并重载——
# 改完 gemm_gpu.cu / gemm_cpu.cpp 后直接重跑本 cell 即可
op.prepare()
verify_result = verify_all(op, backends="gpu")
print(verify_result.to_table())

### Step 2: Benchmark Sweep

方形矩阵扫描 128 → 8192，覆盖已注册实现。

- 后端选择：默认跟随 `config.TEST_BACKENDS`（"all" / "gpu" / "cpu"，在 `test/config.py` 或第一个 cell 设置）；也可在此单独指定：`benchmark_all(op, SQUARE_SIZES, backends="gpu")` 或 `"cpu"`
- GPU：CUDA Event 计时，固定迭代次数
- CPU：`perf_counter` 计时，自适应迭代次数（每个数据点约 1.5s）
- 带 `max_size` 门控的实现自动跳过过大尺寸（如纯 Python 循环）

In [ ]:
# Define sweep sizes (square matrices)
SQUARE_SIZES = [
    (128, 128, 128),
    (256, 256, 256),
    (512, 512, 512),
    (1024, 1024, 1024),
    (2048, 2048, 2048),
    (4096, 4096, 4096),
    (8192, 8192, 8192),
]

# 自动重编译过期产物——改 gemm_gpu.cu / gemm_cpu.cpp 后直接重跑本 cell 即可
op.prepare()

print("=" * 70)
print("GEMM Benchmark Sweep")
print("=" * 70)

# 默认跟随 config.TEST_BACKENDS；也可单独指定后端：
#   sweep = benchmark_all(op, SQUARE_SIZES, backends="gpu")   # 仅 GPU
#   sweep = benchmark_all(op, SQUARE_SIZES, backends="cpu")   # 仅 CPU
sweep = benchmark_all(op, SQUARE_SIZES, backends="gpu")

In [ ]:
fig = plot_roofline(sweep)
plt.show()

print()
print(sweep.to_table())

# Plot — one row per backend (GPU / CPU)
fig = plot_comparison(
    sweep,
    title="GEMM Performance: hand-written vs cuBLAS / NumPy BLAS (FP32)")
plt.show()